In [ ]:
import os

import numpy as np
import pandas as pd


In [15]:
def calculate_row_average(row):
    values = []
    for col in row.index[6:]:  # start from col_6 onwards
        val = row[col]
        try:
            if pd.notna(val):
                values.append(float(val))
        except:
            continue
    if values:
        return np.mean(values)
    else:
        return np.nan

In [ ]:
# === CONFIGURATION === df = read_emotibit_csv('./wrist/2025-05-12_22-05-22-738924.csv')
input_csv = (
    "./wrist/2025-05-12_22-05-22-738924.csv"  # Change this to your CSV file path
)
output_folder = "parsed_sensor_csvs"

# Create output folder if not exists
os.makedirs(output_folder, exist_ok=True)


In [ ]:
df_raw = pd.read_csv(
    input_csv,
    header=None,
    engine="python",
    sep=",",
    names=[f"col_{i}" for i in range(30)],
    skip_blank_lines=True,
)
df_raw["AverageDataValue"] = df_raw.apply(calculate_row_average, axis=1)


df = df_raw.iloc[:, :7]
df.columns = [
    "EmotiBitTimestamp",
    "PacketNumber",
    "DataLength",
    "TypeTag",
    "ProtocolVersion",
    "DataReliability",
    "Data",
]
df["AverageDataValue"] = df_raw["AverageDataValue"]

# df = df[~df['TypeTag'].isin(['EM', 'RB'])].reset_index(drop=True)


In [13]:
df

,EmotiBitTimestamp,PacketNumber,DataLength,TypeTag,ProtocolVersion,DataReliability,Data
0,230808,35646,1,PI,1,100,178753
1,230808,35647,1,PR,1,100,127702
2,230808,35648,1,PG,1,100,7682
3,230807,35649,1,EA,1,100,0.030251
4,230807,35650,1,EL,1,100,26454.199219
...,...,...,...,...,...,...,...
1026413,6700947,45242,4,GY,1,100,2.350
1026414,6700947,45243,4,GZ,1,100,-1.495
1026415,6700947,45244,4,MX,1,100,10
1026416,6700947,45245,4,MY,1,100,49


In [ ]:
# === PREPROCESS ===
# Some rows have multiple data columns (comma-separated in last column)
df_expanded = df.copy()
df_expanded = df_expanded.dropna(subset=["Data"])

# Split multiple values into separate columns
max_columns = df_expanded["Data"].apply(lambda x: len(str(x).split(","))).max()
data_columns = [f"Data_{i}" for i in range(max_columns)]
df_expanded[data_columns] = df_expanded["Data"].apply(
    lambda x: pd.Series(str(x).split(","))
)

# Keep EmotiBitTimestamp as float
df_expanded["EmotiBitTimestamp"] = pd.to_numeric(
    df_expanded["EmotiBitTimestamp"], errors="coerce"
)


# === FUNCTION: Nearest Timestamp Merge for Multi-Axis Signals ===
def align_axes(df, base_tag, axes=["X", "Y", "Z"]):
    axis_dfs = []
    for axis in axes:
        tag = f"{base_tag}{axis}"
        axis_df = df[df["TypeTag"] == tag][["EmotiBitTimestamp", "Data_0"]].copy()
        axis_df = axis_df.rename(columns={"Data_0": axis})
        axis_df["EmotiBitTimestamp"] = pd.to_numeric(
            axis_df["EmotiBitTimestamp"], errors="coerce"
        )
        axis_dfs.append(axis_df.sort_values("EmotiBitTimestamp"))

    # Start with first axis dataframe
    aligned_df = axis_dfs[0]
    for i, other_axis_df in enumerate(axis_dfs[1:], start=1):
        aligned_df = pd.merge_asof(
            aligned_df.sort_values("EmotiBitTimestamp"),
            other_axis_df.sort_values("EmotiBitTimestamp"),
            on="EmotiBitTimestamp",
            direction="nearest",
            tolerance=0.02,  # 20 ms tolerance (adjust if needed)
        )
    aligned_df = aligned_df.dropna()
    return aligned_df


# === ALIGN AND SAVE EACH MULTI-AXIS SENSOR ===
for sensor in ["A", "G", "M"]:  # A=Accel, G=Gyro, M=Magnetometer
    if all(
        [(sensor + axis) in df_expanded["TypeTag"].values for axis in ["X", "Y", "Z"]]
    ):
        aligned = align_axes(df_expanded, sensor)
        filename = {"A": "Accelerometer", "G": "Gyroscope", "M": "Magnetometer"}[sensor]
        aligned.to_csv(
            os.path.join(output_folder, f"EmotiBit_{filename}_Aligned.csv"), index=False
        )
        print(f"✅ Saved {filename} aligned CSV.")

# === SINGLE CHANNEL SENSORS (EDA, TEMP, etc) ===
single_channel_sensors = ["EA", "EL", "T1", "TH", "HR", "SF", "BI"]
for sensor in single_channel_sensors:
    if sensor in df_expanded["TypeTag"].values:
        sensor_df = df_expanded[df_expanded["TypeTag"] == sensor][
            ["EmotiBitTimestamp", "Data_0"]
        ].rename(columns={"Data_0": sensor})
        sensor_df.to_csv(
            os.path.join(output_folder, f"EmotiBit_{sensor}_Raw.csv"), index=False
        )
        print(f"✅ Saved {sensor} raw CSV.")

print("🎉 Done. All aligned sensor CSVs are in:", output_folder)

ParserError: Error tokenizing data. C error: Expected 7 fields in line 2, saw 11
